In [ ]:
import os
import shutil
from datetime import date
from pathlib import Path
from dotenv import load_dotenv
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip


In [ ]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, ShortType, StringType,
    TimestampType, DateType, DecimalType,
)

SCHEMAS = {
    "usuarios": StructType([
        StructField("id",            IntegerType(),  False),
        StructField("nome",          StringType(),   False),
        StructField("email",         StringType(),   False),
        StructField("data_cadastro", DateType(),     False),
        StructField("pais",          StringType(),   False),
        StructField("created_at",    TimestampType(),False),
    ]),
    "planos": StructType([
        StructField("id",           IntegerType(),    False),
        StructField("nome",         StringType(),     False),
        StructField("preco_mensal", DecimalType(8,2), False),
        StructField("created_at",   TimestampType(),  False),
    ]),
    "artistas": StructType([
        StructField("id",         IntegerType(),  False),
        StructField("nome",       StringType(),   False),
        StructField("pais",       StringType(),   False),
        StructField("created_at", TimestampType(),False),
    ]),
    "albuns": StructType([
        StructField("id",              IntegerType(),  False),
        StructField("artista_id",      IntegerType(),  False),
        StructField("titulo",          StringType(),   False),
        StructField("ano_lancamento",  ShortType(),    False),
        StructField("created_at",      TimestampType(),False),
    ]),
    "musicas": StructType([
        StructField("id",         IntegerType(),  False),
        StructField("album_id",   IntegerType(),  False),
        StructField("artista_id", IntegerType(),  False),
        StructField("genero_id",  IntegerType(),  False),
        StructField("titulo",     StringType(),   False),
        StructField("duracao_ms", IntegerType(),  False),
        StructField("created_at", TimestampType(),False),
    ]),
    "generos": StructType([
        StructField("id",         IntegerType(),  False),
        StructField("nome",       StringType(),   False),
        StructField("created_at", TimestampType(),False),
    ]),
    "playlists": StructType([
        StructField("id",         IntegerType(),  False),
        StructField("usuario_id", IntegerType(),  False),
        StructField("nome",       StringType(),   False),
        StructField("created_at", TimestampType(),False),
    ]),
    "dispositivos": StructType([
        StructField("id",         IntegerType(),  False),
        StructField("usuario_id", IntegerType(),  False),
        StructField("tipo",       StringType(),   False),
        StructField("so",         StringType(),   False),
        StructField("created_at", TimestampType(),False),
    ]),
}


In [ ]:
import os
import shutil
from datetime import date
from pathlib import Path
from dotenv import load_dotenv
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

load_dotenv()

is_docker = os.path.exists("/.dockerenv")

MINIO_HOST = "minio:9000" if is_docker else "localhost:9000"
MINIO_USER = os.getenv("MINIO_ROOT_USER",     "minioadmin")
MINIO_PASS = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

builder = (
    SparkSession.builder
    .appName("bronze-dimensoes")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

cliente_minio = Minio(
    MINIO_HOST,
    access_key=MINIO_USER,
    secret_key=MINIO_PASS,
    secure=False,
)


In [ ]:
buckets = {b.name for b in cliente_minio.list_buckets()}
print("Buckets disponíveis:", buckets)

if "bronze" not in buckets:
    cliente_minio.make_bucket("bronze")
    print("Bucket 'bronze' criado.")


In [ ]:
def download_landing_csv(tabela: str, cliente: Minio, destino_dir: Path) -> Path:
    """Baixa o CSV da Landing (MinIO) para um diretório local."""
    destino_dir.mkdir(parents=True, exist_ok=True)
    destino_arquivo = destino_dir / f"{tabela}.csv"
    cliente.fget_object("landing", f"{tabela}/{tabela}.csv", str(destino_arquivo))
    return destino_arquivo


def upload_delta_para_minio(local_dir: Path, bucket: str, prefix: str,
                            cliente: Minio) -> None:
    """Sobe recursivamente a estrutura Delta (parquet + _delta_log) para o MinIO."""
    for arq in sorted(local_dir.rglob("*")):
        if arq.is_file():
            object_name = f"{prefix}/{arq.relative_to(local_dir)}"
            cliente.fput_object(bucket, object_name, str(arq))
            print(f"  → {object_name}")


In [ ]:
INGESTAO_DATE = str(date.today())  
print(f"ingestao_date = {INGESTAO_DATE}")


In [ ]:
for tabela, schema in SCHEMAS.items():
    print(f"\n{'=' * 55}")
    print(f"Tabela: {tabela}")
    print(f"{'=' * 55}")

    local_csv_dir = Path(f"/tmp/landing_local/{tabela}")
    csv_path = download_landing_csv(tabela, cliente_minio, local_csv_dir)
    print(f"CSV baixado: {csv_path}")

    df = (
        spark.read
        .option("header", "true")
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .option("dateFormat", "yyyy-MM-dd")
        .schema(schema)
        .csv(str(csv_path))
    )

    total = df.count()
    print(f"Registros lidos: {total:,}")

    df = df.withColumn("ingestao_date", F.lit(INGESTAO_DATE).cast("date"))

    local_delta_dir = Path(f"/tmp/bronze/{tabela}")
    if local_delta_dir.exists():
        shutil.rmtree(local_delta_dir)

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("ingestao_date")
        .save(str(local_delta_dir))
    )
    print(f"Delta escrito localmente em {local_delta_dir}")

    print(f"Enviando para MinIO bronze/dimensoes/{tabela}/")
    upload_delta_para_minio(
        local_delta_dir, "bronze", f"dimensoes/{tabela}", cliente_minio
    )
    print(f"{tabela} concluído — {total:,} registros na Bronze.")

print("\nPromoção Landing → Bronze concluída para todas as dimensões.")


In [ ]:
spark.stop()
